# Lending Game — Parts 4 and 5: market simulator and winner's curse

**We are lender 3.** Every applicant receives up to three offers, one per lender, and takes the one
with the lowest *effective* rate: the rate of the applicant's preferred lender counts 2 percentage
points lower (`DISCOUNT` in `src/config.py`). The winning lender is paid the rate it actually
offered, not the discounted one.

This notebook runs each function of `src/simulator.py` on small hand-made examples whose result can
be checked by hand. All the logic lives in `src/simulator.py`; the notebook only calls it. The
method is written up in `report/market_simulator.md`, and `tests/test_simulator.py` checks the same
rules automatically.

Conventions: rates are decimals (`0.05` = 5%), `NaN` = no offer, lenders are numbered 1, 2, 3.

---

## 0. Setup

In [1]:
import sys
sys.path.append("..")  # make src/ importable from analysis/

import numpy as np
import pandas as pd

from src.simulator import (assign_preferences, rates_matrix, choice_weights,
                           choose_lender, simulate_market, winners_curse)

---

## 1. Preferences

Each applicant prefers one of the three lenders, and the game does not tell us who prefers whom.
`assign_preferences` draws it: an exact one-third split over lenders 1, 2, 3, shuffled with a fixed
seed so that the draw is reproducible. Here on six applicants.

In [2]:
applicants = pd.DataFrame({"id": [1, 2, 3, 4, 5, 6]})
with_pref = assign_preferences(applicants, seed=42)
with_pref

,id,pref
0,1,1
1,2,3
2,3,3
3,4,2
4,5,2
5,6,1


Each lender is preferred by exactly two of the six applicants, in shuffled order. Running it
again with `seed=42` gives the same assignment.

---

## 2. Reading the offers

`rates_matrix` turns the offers (a DataFrame with columns `r1`, `r2`, `r3`, or an `(n, 3)` array)
into a numeric array. It stops with an error if a rate is negative or above `MAX_RATE` = 2.0, which
would mean the rates were given in percent (12 instead of 0.12). The cap is not 1.0 because a
break-even rate plus a margin can exceed 100% for applicants whose default probability is close to
0.5.

In [3]:
offers = pd.DataFrame({"r1": [0.05, np.nan], "r2": [0.06, 0.12], "r3": [0.08, 0.09]})
rates_matrix(offers)

array([[0.05, 0.06, 0.08],
       [ nan, 0.12, 0.09]])

One row per applicant, one column per lender. Lender 1 makes no offer to the second applicant,
which stays `NaN`.

---

## 3. The choice rule: `choice_weights`

`choice_weights` applies the rule given each applicant's preferred lender (their "type"): it
subtracts the 2 pp discount from the preferred lender's rate, then gives weight 1 to the lowest
effective rate. Ties are split equally and a missing offer never wins. Each output row is the
probability that the applicant takes lender 1, 2 or 3.

First, the same offers (5%, 6%, 8%) shown to one applicant of each type.

In [4]:
rates = np.array([[0.05, 0.06, 0.08]] * 3)   # same offers for 3 applicants
pref  = np.array([1, 2, 3])                   # one applicant of each type
choice_weights(rates, pref)

array([[1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

- Type 1: effective rates 3%, 6%, 8% → lender 1.
- Type 2: 5%, 4%, 8% → lender 2, whose discount is enough to beat 5%.
- Type 3: 5%, 6%, 6% → lender 1, because 8% − 2% = 6% still loses to 5%.

A tie: the applicant prefers lender 1 at 7% (effective 5%), and lenders 2 and 3 both offer 5%.

In [5]:
choice_weights(np.array([[0.07, 0.05, 0.05]]), [1]) # tie in preference

array([[0.33333333, 0.33333333, 0.33333333]])

All three effective rates are 5%, so the weight is split equally: one third each.

Lender 1, the preferred one, makes no offer (`NaN`), so its discount cannot help it.

In [6]:
choice_weights(np.array([[np.nan, 0.10, 0.30]]), [1]) # choose lowest rate

array([[0., 1., 0.]])

The applicant takes the cheapest remaining offer: lender 2 at 10%.

No lender makes an offer.

In [7]:
choice_weights(np.array([[np.nan, np.nan, np.nan]]), [1]) # no offer

array([[0., 0., 0.]])

A row of zeros: no loan is made.

---

## 4. One draw: `choose_lender`

`choose_lender` turns the weights into a single chosen lender per applicant, breaking any tie at
random. Same offers as the first example of section 3, one applicant of each type.

In [8]:
rng = np.random.default_rng(0)
choose_lender(np.array([[0.05, 0.06, 0.08]] * 3), [1, 2, 3], rng=rng)

array([1, 2, 1])

Lenders 1, 2, 1, as in section 3. There is no tie here, so the random draw does not change the
result.

---

## 5. Simulating a market: `simulate_market`

`simulate_market` runs the choice rule for every applicant and adds the profits, using the observed
defaults. Profit per loan, with `LOAN_AMOUNT` = 10,000: rate × 10,000 if the loan is repaid,
−10,000 if it defaults. Five applicants with hand-picked offers; applicants 2 and 4 default.

### 5.1 Expected mode

`mode="expected"` draws no preferences: it averages the outcome over the three types, each with
weight 1/3. There is no randomness, which makes it the mode for comparing pricing rules.

In [9]:
offers = pd.DataFrame(
    {"r1": [0.05, 0.10, np.nan, 0.20, np.nan],
     "r2": [0.06, 0.07, 0.12,   np.nan, np.nan],
     "r3": [0.08, 0.07, 0.09,   np.nan, np.nan]},
    index=[1, 2, 3, 4, 5])
defaults = [0, 1, 0, 1, 0]

loans, summary = simulate_market(offers, defaults, mode="expected")
loans

,r1,r2,r3,default,w1,w2,w3,profit1,profit2,profit3,exp_rate_taken,loan_made_prob
1,0.05,0.06,0.08,0,0.666667,0.333333,0.0,333.333333,200.0,0.0,0.053333,1.0
2,0.10,0.07,0.07,1,0.000000,0.500000,0.5,-0.000000,-5000.0,-5000.0,0.070000,1.0
3,NaN,0.12,0.09,0,0.000000,0.000000,1.0,0.000000,0.0,900.0,0.090000,1.0
4,0.20,NaN,NaN,1,1.000000,0.000000,0.0,-10000.000000,-0.0,-0.0,0.200000,1.0
5,NaN,NaN,NaN,0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0


The weights `w1`–`w3` are now fractions: the share of the three types that take each lender.

- Applicant 1: types 1 and 3 take lender 1, type 2 takes lender 2 → 2/3, 1/3, 0.
- Applicant 2: lenders 2 and 3 tie for type 1 and each wins its own type → 1/2, 1/2. The loan
  defaults, so each loses half of 10,000.
- Applicant 3: lender 3 wins for every type, since 9% beats lender 2's 12% − 2% = 10%.
- Applicant 4: only lender 1 offers; the loan defaults, a loss of 10,000.
- Applicant 5: no offer, no loan (`loan_made_prob` = 0).

The same results, per lender.

In [10]:
summary

,n_loans,share_of_applicants,share_of_loans,avg_rate,default_rate_won,total_profit,profit_per_loan
lender,,,,,,,
1,1.666667,0.333333,0.416667,0.140000,0.600000,-9666.666667,-5800.000000
2,0.833333,0.166667,0.208333,0.066000,0.600000,-4800.000000,-5760.000000
3,1.500000,0.300000,0.375000,0.083333,0.333333,-4100.000000,-2733.333333


`n_loans` adds up the weights, so it can be fractional. Lender 1's average rate of 14% mixes two
thirds of a loan at 5% with a whole loan at 20%. All three lenders lose money here, because two of
the four loans made default.

### 5.2 Random mode

`mode="random"` draws one preferred lender per applicant (with `assign_preferences`, seed 42), so
each applicant goes to exactly one lender. This is one possible outcome, and it changes with the
seed.

In [11]:
loans2, summary2 = simulate_market(offers, defaults, mode="random")
loans2

,r1,r2,r3,default,w1,w2,w3,profit1,profit2,profit3,exp_rate_taken,loan_made_prob,pref,chosen_lender
1,0.05,0.06,0.08,0,0.0,1.0,0.0,0.0,600.0,0.0,0.06,1.0,2,2
2,0.10,0.07,0.07,1,0.0,0.0,1.0,-0.0,-0.0,-10000.0,0.07,1.0,3,3
3,NaN,0.12,0.09,0,0.0,0.0,1.0,0.0,0.0,900.0,0.09,1.0,1,3
4,0.20,NaN,NaN,1,1.0,0.0,0.0,-10000.0,-0.0,-0.0,0.20,1.0,2,1
5,NaN,NaN,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,1,0


The weights are now 0 or 1, and `pref` and `chosen_lender` show the draw. Applicant 1 prefers
lender 2 and takes its 6% (4% effective) over lender 1's 5%. Applicant 3 prefers lender 1, which
made no offer, so lender 3's 9% wins.

The same results, per lender.

In [12]:
summary2

,n_loans,share_of_applicants,share_of_loans,avg_rate,default_rate_won,total_profit,profit_per_loan
lender,,,,,,,
1,1.0,0.2,0.25,0.20,1.0,-10000.0,-10000.0
2,1.0,0.2,0.25,0.06,0.0,600.0,600.0
3,2.0,0.4,0.50,0.08,0.5,-9100.0,-4550.0


Whole numbers of loans this time. Compared with the expected mode, lender 2 now has a single
repaid loan at 6%, and lender 3 carries all of applicant 2's default.

---

## 6. Winner's curse: `winners_curse`

On the loans we (lender 3) win, compare our predicted default probability with the observed default
rate: `gap` = observed − predicted, so a positive gap means we under-estimated the risk of the loans
we won. `p3` stands in for our model's predictions on the five applicants. The applicants are split
into two groups by `p3` (`groups=2`), and the weights come from the expected mode of section 5.1.

In [13]:
p3 = np.array([0.10, 0.30, 0.05, 0.40, 0.20])   # our predicted p for applicants 1 to 5
winners_curse(loans, p3, lender=3, groups=2)

,group_size,n_loans_won,win_rate,avg_pred_p,obs_default_rate,gap,se_obs,p_low,p_high
G1,3,1.0,0.333333,0.050000,0.000000,-0.05,0.000000,0.05,0.2
G2,2,0.5,0.250000,0.300000,1.000000,0.70,0.000000,0.30,0.4
overall,5,1.5,0.300000,0.133333,0.333333,0.20,0.351364,0.05,0.4


- G1 (the three lowest `p3`): we win one whole loan, applicant 3, predicted 5% and repaid → gap
  −5 pp.
- G2 (the two highest): we win half of applicant 2's loan, predicted 30% and defaulted → gap
  +70 pp.
- Overall: 1.5 loans, predicted 13.3%, observed 33.3% → gap +20 pp.

With five applicants these numbers only illustrate the mechanics; `se_obs` gives the approximate
standard error of each observed rate. In Part 6, `p3` will be our model's out-of-sample
predictions.